**Hyperparameter Tuning of Multiple Machine Learning Models using GridSearchCV**

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score

In [2]:
X_train = pd.read_csv("X_train_processed.csv")
X_test = pd.read_csv("X_test_processed.csv")

y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

In [3]:
models = {

    "Logistic Regression": (
        LogisticRegression(max_iter=1000),
        {
            "C":[0.01,0.1,1,10]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {
            "max_depth":[5,10,20,None],
            "min_samples_split":[2,5,10]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators":[50,100,200],
            "max_depth":[5,10,None]
        }
    ),

    "KNN": (
        KNeighborsClassifier(),
        {
            "n_neighbors":[3,5,7,9],
            "weights":["uniform","distance"]
        }
    ),

    "SVM": (
        SVC(probability=True),
        {
            "C":[0.1,1,10],
            "kernel":["linear","rbf"]
        }
    ),

    "Naive Bayes": (
        GaussianNB(),
        {
            "var_smoothing":[1e-09,1e-08,1e-07]
        }
    )

}

In [4]:
results = []

best_accuracy = 0

best_model = None

best_model_name = ""

In [5]:
for name, (model, params) in models.items():

    print("="*60)

    print(name)

    grid = GridSearchCV(

        estimator=model,

        param_grid=params,

        cv=5,

        scoring="accuracy",

        n_jobs=-1

    )

    grid.fit(X_train, y_train)

    predictions = grid.best_estimator_.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    print("Best Parameters")

    print(grid.best_params_)

    print()

    print("Cross Validation Score")

    print(grid.best_score_)

    print()

    print("Testing Accuracy")

    print(accuracy)

    print()

    results.append([

        name,

        grid.best_params_,

        grid.best_score_,

        accuracy

    ])

    joblib.dump(

        grid.best_estimator_,

        name.replace(" ","_").lower() + "_tuned.pkl"

    )

    if accuracy > best_accuracy:

        best_accuracy = accuracy

        best_model = grid.best_estimator_

        best_model_name = name

Logistic Regression
Best Parameters
{'C': 0.01}

Cross Validation Score
1.0

Testing Accuracy
1.0

Decision Tree
Best Parameters
{'max_depth': 5, 'min_samples_split': 2}

Cross Validation Score
0.996875

Testing Accuracy
1.0

Random Forest
Best Parameters
{'max_depth': 5, 'n_estimators': 50}

Cross Validation Score
1.0

Testing Accuracy
1.0

KNN
Best Parameters
{'n_neighbors': 3, 'weights': 'uniform'}

Cross Validation Score
0.99375

Testing Accuracy
1.0

SVM
Best Parameters
{'C': 0.1, 'kernel': 'linear'}

Cross Validation Score
1.0

Testing Accuracy
1.0

Naive Bayes
Best Parameters
{'var_smoothing': 1e-09}

Cross Validation Score
0.9625

Testing Accuracy
0.95



In [6]:
comparison = pd.DataFrame(

    results,

    columns=[

        "Model",

        "Best Parameters",

        "CV Accuracy",

        "Test Accuracy"

    ]

)

print(comparison)

                 Model                           Best Parameters  CV Accuracy  \
0  Logistic Regression                               {'C': 0.01}     1.000000   
1        Decision Tree  {'max_depth': 5, 'min_samples_split': 2}     0.996875   
2        Random Forest      {'max_depth': 5, 'n_estimators': 50}     1.000000   
3                  KNN  {'n_neighbors': 3, 'weights': 'uniform'}     0.993750   
4                  SVM            {'C': 0.1, 'kernel': 'linear'}     1.000000   
5          Naive Bayes                  {'var_smoothing': 1e-09}     0.962500   

   Test Accuracy  
0           1.00  
1           1.00  
2           1.00  
3           1.00  
4           1.00  
5           0.95  


In [7]:
comparison.to_csv(

    "Hyperparameter_Tuning_Results.csv",

    index=False

)

print("Comparison table saved.")

Comparison table saved.


In [8]:
joblib.dump(

    best_model,

    "best_tuned_model.pkl"

)

print()

print("Best Tuned Model")

print(best_model_name)

print("Accuracy")

print(best_accuracy)


Best Tuned Model
Logistic Regression
Accuracy
1.0
